# 07.XGB模型优化

复用 04 输出数据，使用 XGBoost 对比 LGBM。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

import xgboost as xgb
import joblib

In [ ]:
model_type = "含评分_4A"  # 不含评分 / 含评分_4A / 含评分_4B

if model_type == "不含评分":
    train_file = OUTPUT_DIR / f"4.train_valid_data_不含评分_{customer_type}_{target_type}.csv"
    oot_file = OUTPUT_DIR / f"4.oot_data_不含评分_{customer_type}_{target_type}.csv"
elif model_type == "含评分_4A":
    train_file = OUTPUT_DIR / f"4A.train_valid_data_含评分_{customer_type}_{target_type}.csv"
    oot_file = OUTPUT_DIR / f"4A.oot_data_含评分_{customer_type}_{target_type}.csv"
elif model_type == "含评分_4B":
    train_file = OUTPUT_DIR / f"4B.train_valid_data_含评分_{customer_type}_{target_type}.csv"
    oot_file = OUTPUT_DIR / f"4B.oot_data_含评分_{customer_type}_{target_type}.csv"

df_train = pd.read_csv(train_file).fillna(-999)
df_oot = pd.read_csv(oot_file).fillna(-999)

col_var = [c for c in df_train.columns if c not in join_keys + ["flag", target]]

X_train = df_train[df_train["flag"] == "train"][col_var]
y_train = df_train[df_train["flag"] == "train"][target]
X_valid = df_train[df_train["flag"] == "valid"][col_var]
y_valid = df_train[df_train["flag"] == "valid"][target]
X_oot = df_oot[col_var]
y_oot = df_oot[target]

print(X_train.shape, X_valid.shape, X_oot.shape)

In [ ]:
params = {
    "n_estimators": 800,
    "max_depth": 3,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 10,
    "reg_lambda": 40,
    "min_child_weight": 80,
    "gamma": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "n_jobs": 7,
    "random_state": 2024,
    "use_label_encoder": False
}

model_xgb = xgb.XGBClassifier(**params)
model_xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

In [ ]:
y_train_pred = model_xgb.predict_proba(X_train)[:, 1]
y_valid_pred = model_xgb.predict_proba(X_valid)[:, 1]
y_oot_pred = model_xgb.predict_proba(X_oot)[:, 1]

result = pd.concat([
    auc_gini_ks(y_train_pred, y_train, "train"),
    auc_gini_ks(y_valid_pred, y_valid, "valid"),
    auc_gini_ks(y_oot_pred, y_oot, "oot")
])

display(result)

In [ ]:
save_tag = f"XGB_{model_type}_{customer_type}_{target_type}"

joblib.dump(model_xgb, OUTPUT_DIR / f"7.model-{save_tag}.pkl")
result.to_csv(OUTPUT_DIR / f"7.模型效果_{save_tag}.csv", index=False)

fea_imp = pd.DataFrame({
    "feature_name": col_var,
    "feature_importance": model_xgb.feature_importances_
}).sort_values("feature_importance", ascending=False)

fea_imp.to_csv(OUTPUT_DIR / f"7.特征重要性_{save_tag}.csv", index=False)
display(fea_imp.head(20))